In [ ]:
!pip install deep-translator indic-transliteration

In [ ]:
import pandas as pd
import re

# 1. Load the file
file_path = '/content/sen_1k.csv'
df = pd.read_csv(file_path)

# 2. Cleaning Function
def clean_and_filter(text):
    if pd.isna(text): return None # Khali rows handle karne ke liye
    if r'\bn' in str(text): return None # Bangla skip karne ke liye

    # Language tags (\en, \hi) hatane ke liye
    cleaned_text = re.sub(r'\\\w+', '', str(text))
    return cleaned_text.strip()

# 3. Apply Cleaning
df['cleaned_text'] = df['text'].apply(clean_and_filter)

# 4. Drop filtered rows
df = df.dropna(subset=['cleaned_text'])

# 5. Mapping Labels (Negative -> 1, Others -> 0)
# Hum .str.lower() use kar rahe hain taaki 'Negative' aur 'negative' dono handle ho jayein
label_map = {'negative': 1, 'neutral': 0, 'positive': 0}
df['final_label'] = df['label'].str.lower().map(label_map)

# 6. Final Clean up: Remove any rows where mapping failed
df = df.dropna(subset=['final_label'])

# 7. Save to CSV
df[['cleaned_text', 'final_label']].to_csv('Cleaned_Hinglish.csv', index=False)

print(f"Total rows cleaned: {len(df)}")
print("Cleaned_Hinglish.csv is ready for your Major Project!")

In [ ]:
import pandas as pd
import re

# 1. Load the file
df = pd.read_csv('/content/sen_1k.csv')

def clean_text_refined(text):
    if pd.isna(text): return None

    # Bangla rows check (\bn tag) - isse skip karenge
    if r'\bn' in str(text): return None

    # Step A: Identify if it contains Devanagari (Hindi Script)
    # Range \u0900-\u097F covers Devanagari characters
    has_hindi_script = bool(re.search(r'[\u0900-\u097F]', str(text)))

    # Step B: Clean language tags (\en, \hi, \univ)
    cleaned = re.sub(r'\\\w+', '', str(text))
    cleaned = cleaned.strip()

    return cleaned

# 2. Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_text_refined)
df = df.dropna(subset=['cleaned_text'])

# 3. Label Mapping
label_map = {'negative': 1, 'neutral': 0, 'positive': 0}
df['final_label'] = df['label'].str.lower().map(label_map)
df = df.dropna(subset=['final_label'])

# 4. Save
df[['cleaned_text', 'final_label']].to_csv('Cleaned_Hindi_Hinglish.csv', index=False)

print(f"Total Combined Rows (Hindi + Hinglish): {len(df)}")

In [ ]:
import zipfile
import os

zip_path = '/content/Combined Data.csv (2).zip'
extract_path = '/content/extracted_data/'

# Zip file ko extract kar rahe hain
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Check karte hain ki extracted folder mein kaunsi file hai
extracted_files = os.listdir(extract_path)
print("Extracted files:", extracted_files)

# CSV file ka path set karte hain (maan lete hain file ka naam 'Combined Data.csv' hai)
csv_file_path = os.path.join(extract_path, extracted_files[0])

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
import time
from tqdm import tqdm

# 1. Load the extracted CSV
file_path = '/content/extracted_data/Combined Data.csv'
df_eng = pd.read_csv(file_path)

# 2. Automatically detect columns
# Kaggle datasets mein aksar 'statement' aur 'status' hote hain
text_col = 'statement' if 'statement' in df_eng.columns else 'text'
label_col = 'status' if 'status' in df_eng.columns else 'label'

print(f"Using columns: Text='{text_col}', Label='{label_col}'")

# Major Project ke liye hum pehle 1000 rows ka synthetic data banayenge
df_subset = df_eng.head(1000)

synthetic_data = []

print("Generating Synthetic Hindi & Hinglish... (Progress bar niche dekhein)")

# 3. Translation & Transliteration Loop
for index, row in tqdm(df_subset.iterrows(), total=df_subset.shape[0]):
    try:
        text_en = str(row[text_col])
        current_label = row[label_col]

        # A. English -> Hindi (Devanagari)
        hindi_text = GoogleTranslator(source='en', target='hi').translate(text_en)

        # B. Hindi -> Hinglish (Romanization)
        hinglish_text = transliterate(hindi_text, sanscript.DEVANAGARI, sanscript.ITRANS)

        # Dono rows add kar rahe hain
        synthetic_data.append({'text': hindi_text, 'label': current_label, 'source': 'synthetic_hindi'})
        synthetic_data.append({'text': hinglish_text, 'label': current_label, 'source': 'synthetic_hinglish'})

        # Delay to prevent API ban (Very Important)
        if index % 15 == 0:
            time.sleep(1)

    except Exception as e:
        continue

# 4. Save Synthetic Results
df_synthetic = pd.DataFrame(synthetic_data)
df_synthetic.to_csv('Synthetic_Final_Data.csv', index=False)

print(f"\nSuccess! Total Synthetic rows generated: {len(df_synthetic)}")

In [ ]:
import pandas as pd

# 1. Load the datasets
df_suicide = pd.read_csv('/content/Suicide_Detection.csv')
df_reddit = pd.read_csv('/content/reddit_depression_suicidewatch.csv')

# 2. Standardization Logic
# Distress/Risk = 1, Normal = 0
def standardize_label(label):
    label = str(label).lower().strip()
    # Suicide, SuicideWatch, aur Depression sabko 'Distress' (1) maante hain
    if label in ['suicide', 'suicidewatch', 'depression']:
        return 1
    # Non-suicide ya baki sab 'Normal' (0)
    elif label in ['non-suicide', 'none', 'neutral']:
        return 0
    else:
        return 0 # Default fallback

# 3. Apply Mapping
# Suicide_Detection.csv mein column ka naam aksar 'class' hota hai
label_col_1 = 'class' if 'class' in df_suicide.columns else 'label'
df_suicide['final_label'] = df_suicide[label_col_1].apply(standardize_label)

# reddit_depression_suicidewatch.csv mein column check karein
label_col_2 = 'label' if 'label' in df_reddit.columns else 'class'
df_reddit['final_label'] = df_reddit[label_col_2].apply(standardize_label)

# 4. Filter relevant columns and Combine
# Hum sirf 'text' aur 'final_label' columns hi rakhenge
df_suicide_clean = df_suicide[['text', 'final_label']].rename(columns={'final_label': 'label'})
df_reddit_clean = df_reddit[['text', 'final_label']].rename(columns={'final_label': 'label'})

# Combine English data
df_english_master = pd.concat([df_suicide_clean, df_reddit_clean], ignore_index=True)

# 5. Remove any duplicates or empty rows
df_english_master = df_english_master.dropna().drop_duplicates(subset=['text'])

print("Class Distribution After Standardization:")
print(df_english_master['label'].value_counts())
print(f"\nTotal English records ready: {len(df_english_master)}")

# Ek chota sample dekhne ke liye
print("\nSample Data:")
print(df_english_master.head())

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
import time
from tqdm import tqdm

# 1. Sabse pehle 1500 'Suicide' aur 1500 'Non-Suicide' ka sample lete hain
df_distress = df_english_master[df_english_master['label'] == 1].sample(1500, random_state=42)
df_normal = df_english_master[df_english_master['label'] == 0].sample(1500, random_state=42)
df_to_translate = pd.concat([df_distress, df_normal]).sample(frac=1) # Shuffle

synthetic_results = []

print("Starting Synthetic Generation for 3,000 rows...")

# 2. Translation & Transliteration Loop
for index, row in tqdm(df_to_translate.iterrows(), total=df_to_translate.shape[0]):
    try:
        # Long text ko short karte hain taaki translation fast ho
        text_en = str(row['text'])[:400]
        lbl = row['label']

        # English -> Hindi
        hindi_text = GoogleTranslator(source='en', target='hi').translate(text_en)

        # Hindi -> Hinglish (Roman)
        hinglish_text = transliterate(hindi_text, sanscript.DEVANAGARI, sanscript.ITRANS)

        synthetic_results.append({'text': hindi_text, 'label': lbl, 'lang': 'hindi'})
        synthetic_results.append({'text': hinglish_text, 'label': lbl, 'lang': 'hinglish'})

        # Batch delay to avoid API block
        if index % 25 == 0:
            time.sleep(1)

    except Exception as e:
        continue

# 3. Save Synthetic Data
df_synthetic = pd.DataFrame(synthetic_results)
df_synthetic.to_csv('Suicide_Hindi_Hinglish_Synthetic.csv', index=False)
print(f"\nDone! Generated {len(df_synthetic)} Multilingual rows.")

In [ ]:
os.listdir("/content")

In [ ]:
import pandas as pd

# 1. Load All Datasets
df_suicide_en = pd.read_csv('Suicide_Detection.csv')
df_reddit_en = pd.read_csv('reddit_depression_suicidewatch.csv')
df_hindi_sent = pd.read_csv('hindi_sentiment_dataset.csv')
df_real_mix = pd.read_csv('Cleaned_Hindi_Hinglish.csv')
df_synth_hi = pd.read_csv('Suicide_Hindi_Hinglish_Synthetic.csv')

# --- Helper Function to detect correct column and clean ---
def clean_df_smart(df, label_val, possible_text_cols=['text', 'statement', 'body'], possible_label_cols=['label', 'sentiment', 'class', 'status']):
    # Find text column
    text_col = next((c for c in possible_text_cols if c in df.columns), df.columns[0])
    # Find label column
    label_col = next((c for c in possible_label_cols if c in df.columns), None)

    temp = df.copy()
    if label_col:
        # Filtering logic specific to your task
        if label_val == 1: # Depression
            temp = temp[temp[label_col].astype(str).str.lower().isin(['depression', 'negative'])]
        elif label_val == 2: # Suicidal
            temp = temp[temp[label_col].astype(str).str.lower().isin(['suicide', 'suicidewatch'])]
        elif label_val == 0: # Normal
            temp = temp[temp[label_col].astype(str).str.lower().isin(['non-suicide', 'neutral', 'positive'])]

    res = temp[[text_col]].rename(columns={text_col: 'text'})
    res['label'] = label_val
    return res.dropna().drop_duplicates()

# 2. Creating Balanced Classes
target_per_class = 10000 # Memory limit ke liye 10k rakhte hain pehle

# --- CLASS 0: NORMAL ---
class_0 = clean_df_smart(df_suicide_en, 0).sample(min(target_per_class, 50000), random_state=42).head(target_per_class)

# --- CLASS 1: DEPRESSION ---
dep_en = clean_df_smart(df_reddit_en, 1)
dep_hi = clean_df_smart(df_hindi_sent, 1)
class_1 = pd.concat([dep_en, dep_hi]).sample(min(target_per_class, 40000), random_state=42).head(target_per_class)

# --- CLASS 2: SUICIDAL ---
sui_en = clean_df_smart(df_suicide_en, 2)
sui_reddit = clean_df_smart(df_reddit_en, 2)
# Synthetic data mein distress label 1 tha, hum use class 2 (Suicidal) maan rahe hain
sui_synth = df_synth_hi[['text']].copy()
sui_synth['label'] = 2

class_2 = pd.concat([sui_en, sui_reddit, sui_synth]).sample(min(target_per_class, 40000), random_state=42).head(target_per_class)

# 3. Final Master Merge
master_multiclass = pd.concat([class_0, class_1, class_2], ignore_index=True)
master_multiclass = master_multiclass.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Save
master_multiclass.to_csv('FINAL_MAJOR_PROJECT_MASTER.csv', index=False)

print("✅ Master Dataset Ready without Errors!")
print(master_multiclass['label'].value_counts())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Load Sources
df_suicide_en = pd.read_csv('Suicide_Detection.csv')
df_reddit_en = pd.read_csv('reddit_depression_suicidewatch.csv')
df_synth_hi = pd.read_csv('Suicide_Hindi_Hinglish_Synthetic.csv')

target_per_class = 8000

# 1. Class 0: Normal
class_0 = df_suicide_en[df_suicide_en['class'] == 'non-suicide'].copy().head(target_per_class)
class_0 = class_0[['text']].assign(label=0).reset_index(drop=True)

# 2. Class 1: Depression
class_1 = df_reddit_en[df_reddit_en['label'] == 'depression'].copy().head(target_per_class)
class_1 = class_1.rename(columns={'text': 'text'}).assign(label=1).reset_index(drop=True)

# 3. Class 2: Suicidal
class_2 = df_suicide_en[df_suicide_en['class'] == 'suicide'].copy().head(target_per_class)
class_2 = class_2[['text']].assign(label=2).reset_index(drop=True)

# 4. Class 3: Anxiety (Keywords based)
anx_kw = ['anxiety', 'anxious', 'panic', 'bechaini', 'dar']
class_3 = df_suicide_en[df_suicide_en['text'].str.contains('|'.join(anx_kw), case=False, na=False)].head(target_per_class)
class_3 = class_3[['text']].assign(label=3).reset_index(drop=True)

# 5. Class 4: Stress (Keywords based)
str_kw = ['stress', 'pressure', 'exhausted', 'exams', 'tension', 'bojh']
class_4 = df_reddit_en[df_reddit_en['text'].str.contains('|'.join(str_kw), case=False, na=False)].head(target_per_class)
class_4 = class_4.rename(columns={'text': 'text'}).assign(label=4).reset_index(drop=True)

print("Classes extracted successfully!")

In [ ]:
# Merging all 5 classes
dfs = [class_0, class_1, class_2, class_3, class_4]
master_5class = pd.concat(dfs, ignore_index=True)

# Cleaning & Shuffling
master_5class = master_5class.dropna(subset=['text'])
master_5class = master_5class.drop_duplicates(subset=['text'])
master_5class = master_5class.sample(frac=1, random_state=42).reset_index(drop=True)

# Saving to Google Drive
# Make sure you have a folder named 'Major_Project' in your Drive or change path
drive_path = '/content/drive/MyDrive/FINAL_5CLASS_MASTER_DATASET.csv'
master_5class.to_csv(drive_path, index=False)

print(f"✅ Dataset saved safely to Drive at: {drive_path}")
print("\nFinal Class Counts:")
print(master_5class['label'].value_counts().sort_index())

In [ ]:
import pandas as pd

# Keywords for Stress
stress_extended_kw = ['stress', 'pressure', 'exhausted', 'exams', 'tension', 'bojh', 'workload',
                      'tired', 'burnout', 'overwhelmed', 'preshan', 'thak gaya', 'tension', 'ghabrahat']

def get_text_col(df):
    """Apne aap text column ka naam dhoondne ke liye"""
    possible_cols = ['text', 'statement', 'body', 'content', 'message']
    for col in possible_cols:
        if col in df.columns:
            return col
    return df.columns[0] # Agar kuch na mile toh pehla column le lo

# 1. English Sources se Stress nikalna
col_sui = get_text_col(df_suicide_en)
str_en_1 = df_suicide_en[df_suicide_en[col_sui].astype(str).str.contains('|'.join(stress_extended_kw), case=False, na=False)].copy()
str_en_1 = str_en_1[[col_sui]].rename(columns={col_sui: 'text'})

col_red = get_text_col(df_reddit_en)
str_en_2 = df_reddit_en[df_reddit_en[col_red].astype(str).str.contains('|'.join(stress_extended_kw), case=False, na=False)].copy()
str_en_2 = str_en_2[[col_red]].rename(columns={col_red: 'text'})

# 2. Hindi/Hinglish Mixed Source (Jahan error aa raha tha)
df_mix = pd.read_csv('Cleaned_Hindi_Hinglish.csv')
col_mix = get_text_col(df_mix)
str_hi = df_mix[df_mix[col_mix].astype(str).str.contains('|'.join(stress_extended_kw), case=False, na=False)].copy()
str_hi = str_hi[[col_mix]].rename(columns={col_mix: 'text'})

# 3. Combine and Label as 4
class_4_updated = pd.concat([str_en_1, str_en_2, str_hi], ignore_index=True)
class_4_updated['label'] = 4
class_4_updated = class_4_updated.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"✅ Success! Stress Class (Label 4) updated. New Row Count: {len(class_4_updated)}")

In [ ]:
import pandas as pd

# Target rows per class for a perfectly balanced 5-class model
final_target = 8000

# Balancing each class
# We use .sample to pick random rows if we have more than target,
# otherwise we take all available rows.
b0 = class_0.sample(min(final_target, len(class_0)), random_state=42)
b1 = class_1.sample(min(final_target, len(class_1)), random_state=42)
b2 = class_2.sample(min(final_target, len(class_2)), random_state=42)
b3 = class_3.sample(min(final_target, len(class_3)), random_state=42)
b4 = class_4_updated.sample(min(final_target, len(class_4_updated)), random_state=42)

# Merge all into one Master DataFrame
master_final = pd.concat([b0, b1, b2, b3, b4], ignore_index=True)

# Shuffle the data (Crucial so the model doesn't learn the order)
master_final = master_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to Google Drive
save_path = '/content/drive/MyDrive/MAJOR_PROJECT_5CLASS_BALANCED.csv'
master_final.to_csv(save_path, index=False)

print(f"✅ Final Balanced Master Dataset Saved to Drive: {save_path}")
print("-" * 30)
print("Final Class Distribution (The Model's World View):")
print(master_final['label'].value_counts().sort_index())

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Verify the file exists
file_path = '/content/drive/MyDrive/MAJOR_PROJECT_5CLASS_BALANCED.csv'
if os.path.exists(file_path):
    print("✅ Balanced Dataset found! Ready to train.")
else:
    print("❌ Dataset not found. Please check your Drive path.")

In [ ]:
!pip install transformers[torch] datasets evaluate scikit-learn

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# 1. Load Dataset from Drive
dataset = load_dataset('csv', data_files=file_path)
dataset = dataset['train'].train_test_split(test_size=0.15, seed=42)

# 2. Use MuRIL Tokenizer
model_nm = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_nm)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_func, batched=True)

# 3. Initialize MuRIL for 5 Classes
# Labels: 0:Normal, 1:Depression, 2:Suicidal, 3:Anxiety, 4:Stress
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=5)

In [ ]:
# Metric for Evaluation
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Training Arguments (Latest Version Compatible)
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MuRIL_MentalHealth_Checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

# Trainer Initialization (Removed 'tokenizer' argument)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 🚀 START TRAINING
print("🚀 Training Started... MuRIL is processing your 5 classes.")
trainer.train()

# Final Save to Drive
model.save_pretrained("/content/drive/MyDrive/MuRIL_MentalHealth_Final")
tokenizer.save_pretrained("/content/drive/MyDrive/MuRIL_MentalHealth_Final")


print("✅ TRAINING COMPLETE! Your MuRIL model is saved to Google Drive.")

In [ ]:
from transformers import pipeline

# 1. Load the saved model and tokenizer from Drive
model_path = "/content/drive/MyDrive/MuRIL_MentalHealth_Final"
classifier = pipeline("text-classification", model=model_path, tokenizer=model_path)

# 2. Label Mapping
labels = {
    "LABEL_0": "Normal",
    "LABEL_1": "Depression",
    "LABEL_2": "Suicidal",
    "LABEL_3": "Anxiety",
    "LABEL_4": "Stress"
}

def predict_mental_health(text):
    result = classifier(text)[0]
    category = labels[result['label']]
    score = result['score']
    print(f"Text: {text}")
    print(f"Prediction: {category} (Confidence: {score:.2f})")
    print("-" * 30)

# 3. Test Cases
predict_mental_health("Kal mera final exam hai aur mujhe bohot ghabrahat ho rahi hai.")
predict_mental_health("I feel so hopeless and sad every single day.")
predict_mental_health("Office ka workload itna zyada hai ki main thak gaya hoon.")
predict_mental_health("I had a great lunch with my friends today.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Get predictions for the test set
print("Generating predictions for evaluation...")
predictions = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions.predictions, axis=-1)
actual = predictions.label_ids

# 2. Print Classification Report
target_names = ["Normal", "Depression", "Suicidal", "Anxiety", "Stress"]
report = classification_report(actual, preds, target_names=target_names)
print("\n--- Classification Report ---")
print(report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Labels for the axis
target_names = ["Normal", "Depression", "Suicidal", "Anxiety", "Stress"]

# 2. Generate the Matrix
cm = confusion_matrix(actual, preds)

# 3. Plotting
plt.figure(figsize=(9, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=target_names,
            yticklabels=target_names)

plt.title('Confusion Matrix: Mental Health State Classification (MuRIL)', fontsize=15)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

In [ ]:
# Model se raw probabilities (logits) nikalna
predictions = trainer.predict(tokenized_datasets["test"])
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()

In [ ]:
# Example: Increasing threshold for Suicidal (Label 2)
threshold_suicide = 0.75

final_tuned_preds = []
for p in probs:
    # Agar Suicidal class ki probability threshold se kam hai
    if p[2] < threshold_suicide:
        # Toh Suicidal ko chhod kar baaki 4 classes mein se max choose karo
        p[2] = 0 # Temporary ignore suicidal
        final_tuned_preds.append(np.argmax(p))
    else:
        final_tuned_preds.append(np.argmax(p))

In [ ]:
from sklearn.metrics import classification_report

print("--- Classification Report AFTER Threshold Tuning ---")
# final_tuned_preds wo hain jo aapne tuning code se nikaale hain
print(classification_report(actual, final_tuned_preds, target_names=target_names))

In [ ]:
# Function to use tuned logic
def predict_tuned(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()[0]

    # Apply your threshold
    if probs[2] < 0.75: # Your Suicidal Threshold
        probs[2] = 0
        final_idx = np.argmax(probs)
    else:
        final_idx = 2

    print(f"Text: {text}")
    print(f"Tuned Prediction: {target_names[final_idx]} (Confidence: {probs[final_idx]:.2f})")

# Test again
predict_tuned("Kal mera final exam hai aur mujhe bohot ghabrahat ho rahi hai.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Labels mapping
target_names = ["Normal", "Depression", "Suicidal", "Anxiety", "Stress"]

# 2. Generate Matrix using final_tuned_preds (Tuned results)
cm_tuned = confusion_matrix(actual, final_tuned_preds)

# 3. Plotting the Heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=target_names,
            yticklabels=target_names)

plt.title('CONFUSION MATRIX: After Threshold Tuning (MuRIL)', fontsize=15)
plt.xlabel('Predicted Label (Tuned)', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

In [ ]:
import pandas as pd

# 1. Classification report ko dictionary format mein generate karna
report_dict = classification_report(actual, final_tuned_preds, target_names=target_names, output_dict=True)

# 2. Dictionary ko Pandas DataFrame mein convert karna
df_report = pd.DataFrame(report_dict).transpose()

# 3. Excel file save karna (Drive path par)
excel_path = '/content/drive/MyDrive/MuRIL_Tuned_Report.xlsx'
df_report.to_excel(excel_path)

print(f"✅ Classification Report saved at: {excel_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Matrix generate karna
cm_tuned = confusion_matrix(actual, final_tuned_preds)

# 2. Plotting logic
plt.figure(figsize=(9, 6))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=target_names,
            yticklabels=target_names)

plt.title('Final Confusion Matrix (Tuned MuRIL)', fontsize=15)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)

# 3. Image file save karna (Drive path par)
img_path = '/content/drive/MyDrive/Confusion_Matrix_Tuned.png'
plt.savefig(img_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Confusion Matrix Image saved at: {img_path}")

In [ ]:
!pip install shap
import shap
import torch

In [ ]:
# 1. Define the prediction function for SHAP
def f(x):
    tv = tokenizer(x.tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt").to("cuda")
    outputs = model(**tv)[0]
    scores = torch.nn.functional.softmax(outputs, dim=-1).cpu().detach().numpy()
    return scores

# 2. Initialize the SHAP Explainer
# We use the 'text' masker to handle Hinglish strings properly
explainer = shap.Explainer(f, tokenizer, output_names=target_names)

In [ ]:
# Test sentence
test_text = ["Kal mera final exam hai aur mujhe bohot ghabrahat ho rahi hai."]

# Generate SHAP values
shap_values = explainer(test_text)

# Visualize the explanation
shap.plots.text(shap_values[0, :, "Stress"]) # Change "Stress" to any class to see why it was chosen

In [ ]:
# We take a small batch from your test set to see general trends
samples = dataset['test']['text'][:10]
shap_values_global = explainer(samples)

# Plotting global importance for 'Stress'
plt.figure(figsize=(8,4))
shap.plots.bar(shap_values_global[:,:,"Stress"].mean(0))
plt.title("Top Global Predictors for Stress (XAI)")
plt.savefig('/content/drive/MyDrive/XAI_Global_Importance.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Generate the Global Bar Plot for a specific class (e.g., 'Stress' or 'Suicidal')
# Note: shap_values_global should already be calculated from your previous step
plt.figure(figsize=(8, 4))

# We plot the mean absolute value of SHAP values to see overall importance
shap.plots.bar(shap_values_global[:,:,"Stress"].mean(0), show=False)

# 2. Add a title and labels for your report
plt.title("Global Feature Importance for Stress Detection (MuRIL + XAI)", fontsize=16)

# 3. Save the image to your Google Drive
output_image_path = '/content/drive/MyDrive/Global_XAI_Importance_Plot.png'
plt.savefig(output_image_path, dpi=300, bbox_inches='tight')

# 4. Show it in the notebook
plt.show()

print(f"✅ Global XAI Plot successfully saved to: {output_image_path}")

In [ ]:
import os
if os.path.exists("/root/.cache/huggingface/token"):
    os.remove("/root/.cache/huggingface/token")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Model ko push karein
model.push_to_hub("jyotisangam/muril-mental-health-xai")

# Tokenizer ko push karein
tokenizer.push_to_hub("jyotisangam/muril-mental-health-xai")